In [7]:
from os import environ
from dotenv import load_dotenv

load_dotenv()

True

# Parser

In [8]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

class Parser:
    def __init__(self):
        self.embeddings_client = OpenAIEmbeddings(
            base_url= environ.get("GITHUB_ENDPOINT"),    # 🌐 GitHub Models API endpoint
            api_key= environ.get("GITHUB_TOKEN"),        # 🔑 Authentication token
            model=environ.get("GITHUB_EMBEDDINGS_MODEL_ID")  # 🎯 Selected AI model
        )

    def parse_url(self, url: str, html_tag_classes_filter: tuple[str, ...]) -> list[Document]:
        # Only keep post title, headers, and content from the full HTML.
        # A special class/object that filters HTML documents for relevant data
        bs4_strainer = bs4.SoupStrainer(class_=html_tag_classes_filter)
        # A "Loader" is an object representing the method for grabbing data, in this case, from a public website with sample data
        loader = WebBaseLoader(
            web_paths=(url,),
            bs_kwargs={"parse_only": bs4_strainer},
        )
        docs = loader.load()

        assert len(docs) == 1
        print(f"Total characters: {len(docs[0].page_content)}")

        # Print out first 500 characters from the document
        print(docs[0].page_content[:500])

        text_splitter = SemanticChunker(
            embeddings=self.embeddings_client,
            breakpoint_threshold_type="standard_deviation",
            breakpoint_threshold_amount=1
        )
        all_splits = text_splitter.split_documents(docs)

        print(f"Split blog post into {len(all_splits)} sub-documents.")
        return all_splits

blog_post_splits = Parser().parse_url("https://lilianweng.github.io/posts/2023-06-23-agent/", ("post-title", "post-header", "post-content"))

Total characters: 43047


      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In
Split blog post into 53 sub-documents.


# RAG Builder

In [9]:
# Shoves our document chunks into the embeddings model, and stores them in our local chroma vector store database
from langchain_chroma import Chroma

class RAGBuilder:
    def __init__(self):
        self.embeddings_client = OpenAIEmbeddings(
            base_url= environ.get("GITHUB_ENDPOINT"),    # 🌐 GitHub Models API endpoint
            api_key= environ.get("GITHUB_TOKEN"),        # 🔑 Authentication token
            model=environ.get("GITHUB_EMBEDDINGS_MODEL_ID")  # 🎯 Selected AI model
        )
    
    def create_vector_store(self, store_name: str, documents: list[Document]):
        vector_store = Chroma(
            collection_name=store_name,
            embedding_function=self.embeddings_client,
            persist_directory="./.chroma_db"
        )
        document_ids = vector_store.add_documents(documents)

        print(document_ids[:3])
        return vector_store

blog_post_vector_store = RAGBuilder().create_vector_store("LLM_Powered_Autonomous_Agents", blog_post_splits)

['60d9edaf-0bff-4a8a-8071-11e640a791b2', '9c40449f-2add-4c94-bad0-75801d97a652', '244e95cd-1b79-4ec9-8f13-dfcfdc935717']


# RAG Retriever Agent

In [ ]:
from typing import Annotated
from pydantic import Field
from agent_framework.openai import OpenAIChatClient

class RAGRetrieverChatAgent:
    def __init__(self, vector_store):
        self.vector_store = vector_store
        self.chat_client = OpenAIChatClient(
            base_url= environ.get("GITHUB_ENDPOINT"),    # 🌐 GitHub Models API endpoint
            api_key= environ.get("GITHUB_TOKEN"),        # 🔑 Authentication token
            model_id= environ.get("GITHUB_MODEL_ID")  # 🎯 Selected AI model
        )
        self.agent = self.chat_client.as_agent(
            name= "AIAgentBlogPostAgent",
            instructions= """You are a helpful agent to answer queries about AI Agents.
            Answer only using the provided context from the AI Agents blog post.""",
            tools= [self.tool_get_ai_agents_context]
        )

    def get_agent(self):
        return self.agent

    def tool_get_ai_agents_context(self, query: Annotated[str, Field(description="The query to retrieve context from the AI Agents blog post")]) -> str:
        """Retrieves information from a blog post about AI Agents"""

        # K-nearest neighbours search with Chroma
        retrieved_docs = self.vector_store.similarity_search(query, k=2)

        # The "serialised" result, which is just a long string combining documents (chunks) with their metadata
        serialized = "\n\n".join(
            (f"Source: {doc.metadata}\nContent: {doc.page_content}")
            for doc in retrieved_docs
        )

        print(serialized)

        return serialized, retrieved_docs

blog_post_agent = RAGRetrieverChatAgent(blog_post_vector_store).get_agent()

In [11]:
from agent_framework import WorkflowBuilder

workflow = WorkflowBuilder(start_executor=blog_post_agent).build()

In [12]:
result = await workflow.run("What is the standard method for Task Decomposition?")

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 2578}
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a PDDL plan based on an existing “Domain PDDL”, and finally (3) translates the PDDL plan back into natural language. Essentially, the planning step is outsourced to an external tool, assuming the availability of domain-specific PDDL and a suitable plann

In [13]:
for output in result.get_outputs():
    output = str.replace(output.text, ". ", ".\n")
    print(f"{output}\n")

The standard methods for task decomposition are as follows:

1.
**Prompting with LLM**: Using simple prompts such as "Steps for XYZ" or "What are the subgoals for achieving XYZ?".
2.
**Task-specific Instructions**: Providing specific instructions tailored to the task, for example, "Write a story outline" for writing a novel.
3.
**Human Inputs**: Involvement of human input in the decomposition process.

Additionally, there is a distinct approach called LLM+P that involves using an external classical planner for long-horizon planning.
This method utilizes the Planning Domain Definition Language (PDDL) to describe planning problems, translating the problem into "Problem PDDL," then generating a PDDL plan through a classical planner, and finally translating that plan back into natural language.


This external planning step is common in certain robotic setups but may not be applicable in many other domains.

